### New algorithm for enumerating every $\Z_2^n$-colorable PL~spheres of Picard number $5$

In [1]:
using Oscar
using ProgressBars
using SparseArrays
using Serialization
using Base.Threads

println("Running on $(Threads.nthreads()) threads")



  ___   ___   ___    _    ____
 / _ \ / __\ / __\  / \  |  _ \  | Combining and extending ANTIC, GAP,
| |_| |\__ \| |__  / ^ \ |  ´ /  | Polymake and Singular
 \___/ \___/ \___//_/ \_\|_|\_\  | Type "?Oscar" for more information
o--------o-----o-----o--------o  | Documentation: https://docs.oscar-system.org
  S Y M B O L I C   T O O L S    | Version 1.4.1
Running on 1 threads


In [2]:
function boundary_incidence_facets_to_ridges(facets::Vector{UInt16})
    # d = popcount(facets[1]) - 1  # facet dimension
    # @assert all(popcount(f) == d+1 for f in facets) "Need a pure complex (all facets same size)."

    # collect ridges (each facet contributes its (d-1)-subfaces by deleting one vertex)
    ridge_dict = Dict{UInt16, Int}()  # ridge -> row index
    ridges = Vector{UInt16}()
    for f in facets
        for i=0:((8*sizeof(f))-1)
            if (f>>i)&1==1
                r = f ⊻ (UInt16(1)<<i)
                if !haskey(ridge_dict, r)
                    push!(ridges, r)
                    ridge_dict[r] = length(ridges)
                end
            end
        end
    end

    m = length(ridges)
    n = length(facets)

    # build sparse 0/1 matrix (m x n): ridges x facets
    I = Int[]   # row indices
    J = Int[]   # col indices
    V = Int8[]  # values (all ones)

    for (j, f) in pairs(facets)
        for i=0:(8*sizeof(f)-1)
            if (f>>i)&1==1
                r = f ⊻ (UInt16(1)<<i)
                i = ridge_dict[r]
                push!(I, i)
                push!(J, j)
                push!(V, true)
            end
        end
    end

    A = sparse(I, J, V, m, n)  # SparseMatrixCSC{Bool} 

    return ridges, A
end



boundary_incidence_facets_to_ridges (generic function with 1 method)

In [3]:
function kernel_basis_mod2_sparse(A)
    m, n = size(A)

    # Store rows as sparse BitVectors (using sets of column indices)
    rows = [Set{Int}() for _ in 1:m]
    
    # Build rows from A mod 2
    @inbounds for j in 1:n
        for p in nzrange(A, j)
            i = rowvals(A)[p]
            if isodd(A.nzval[p])
                if j in rows[i]
                    delete!(rows[i], j)  # XOR: 1⊕1=0
                else
                    push!(rows[i], j)     # XOR: 0⊕1=1
                end
            end
        end
    end

    # RREF over GF(2), recording pivot columns and pivot rows
    pivcol = Int[]
    pivrow = Int[]
    row = 1
    
    @inbounds for col in 1:n
        row > m && break

        # Find pivot in this column at/under current row
        piv = 0
        for r in row:m
            if col in rows[r]
                piv = r
                break
            end
        end
        piv == 0 && continue

        # Swap rows
        if piv != row
            rows[row], rows[piv] = rows[piv], rows[row]
        end

        push!(pivcol, col)
        push!(pivrow, row)

        # Eliminate this column in all other rows (RREF)
        pivot_row = rows[row]
        for r in 1:m
            if r != row && col in rows[r]
                # XOR rows[r] with pivot_row
                symdiff!(rows[r], pivot_row)
            end
        end

        row += 1
    end

    # Free columns
    pivot_set = Set(pivcol)
    free_cols = [j for j in 1:n if !(j in pivot_set)]

    basis = BitVector[]
    isempty(free_cols) && return basis

    # Build one kernel vector per free column
    @inbounds for f in free_cols
        x_set = Set{Int}([f])  # Sparse representation of x

        # Back-substitute using RREF rows
        for t in length(pivcol):-1:1
            c = pivcol[t]
            r = pivrow[t]
            
            # Compute parity: count elements in rows[r] ∩ x_set, excluding c
            row_r = rows[r]
            parity = false
            for j in row_r
                if j != c && j in x_set
                    parity = !parity
                end
            end
            
            # Set x[c] based on parity
            if parity
                push!(x_set, c)
            else
                delete!(x_set, c)
            end
        end

        # Convert sparse set to BitVector
        x = falses(n)
        for j in x_set
            x[j] = true
        end
        push!(basis, x)
    end

    return basis
end

kernel_basis_mod2_sparse (generic function with 1 method)

In [4]:
using SparseArrays

"""
    kernel_elements_with_Ax_in_0_or_2_noprune(A, basis) -> Vector{BitVector}

Full enumeration WITHOUT pruning.

- A is strictly 0/1 and used over ℤ for Ax.
- basis is a GF(2)-basis of ker(A mod 2), given as Vector{BitVector}.
- Enumerates all nonzero x in the GF(2)-span of `basis` and returns those with Ax ∈ {0,2}^m.
- Complexity: Θ(2^k * (cost to update/check)), where k = length(basis).
"""
function kernel_elements_with_Ax_in_0_or_2(
    A::SparseMatrixCSC{<:Integer},
    basis::Vector{BitVector},
)
    m, n = size(A)
    k = length(basis)
    # println("Dimension of the kernel: ",k)
    @assert all(length(b) == n for b in basis) "Basis vectors must have length = #cols(A)."

    # column -> incident rows (A is 0/1)
    rv = rowvals(A)
    colrows = Vector{Vector{Int}}(undef, n)
    for j in 1:n
        colrows[j] = [rv[p] for p in nzrange(A, j)]
    end

    # supports of basis vectors (columns they toggle)
    supp = Vector{Vector{Int}}(undef, k)
    for t in 1:k
        supp[t] = findall(basis[t])
    end

    x = falses(n)
    counts = zeros(Int16, m)  # Ax over ℤ
    sols = BitVector[]

    function toggle_basis!(t::Int)
        @inbounds for j in supp[t]
            turning_on = !x[j]
            x[j] = turning_on
            δ = turning_on ? Int16(1) : Int16(-1)
            for i in colrows[j]
                counts[i] += δ
            end
        end
    end

    @inline function feasible_final()::Bool
        @inbounds for i in 1:m
            c = counts[i]
            if !(c == 0 || c == 2)
                return false
            end
        end
        return true
    end

    function dfs(t::Int, any_one::Bool)
        if t > k
            if any_one && feasible_final()
                push!(sols, copy(x))
            end
            return
        end

        # branch 0: do not include basis[t]
        dfs(t + 1, any_one)

        # branch 1: include basis[t]
        toggle_basis!(t)
        dfs(t + 1, true)
        toggle_basis!(t)  # toggle back (self-inverse over GF(2))
    end

    dfs(1, false)  # excludes zero vector by any_one=false
    return sols
end


kernel_elements_with_Ax_in_0_or_2

In [5]:
# If homology is UNREDUCED in your setup (most common):
function is_homology_sphere(K::Oscar.SimplicialComplex)
    b = betti_numbers(K)
    # print(b)
    d = dim(K)
    for i in 1:d
        if i == d
            b[i] == 1 || return false
        else
            b[i] == 0 || return false
        end
    end
    return true
end


is_homology_sphere (generic function with 1 method)

In [30]:
using SparseArrays
using LinearAlgebra

"""
Check if a simplicial complex is a mod 2 homology sphere of dimension d.
A d-dimensional mod 2 homology sphere has:
- H_0 = ℤ/2ℤ (connected, one component)
- H_i = 0 for 0 < i < d
- H_d = ℤ/2ℤ
"""
function is_mod2_homology_sphere(complex, d::Int)
    # Check H_0: should have dimension 1 (one connected component)
    if d >= 0
        dim_H0 = compute_mod2_homology_dim(complex, 0)
        if dim_H0 != 1
            println("H_0 has dimension $dim_H0, expected 1 (not connected)")
            return false
        end
    end
    
    # Check H_i = 0 for 0 < i < d
    for i in 1:(d-1)
        dim_Hi = compute_mod2_homology_dim(complex, i)
        if dim_Hi != 0
            println("H_$i has dimension $dim_Hi, expected 0")
            return false
        end
    end
    
    # Check H_d = ℤ/2ℤ (dimension 1)
    dim_Hd = compute_mod2_homology_dim(complex, d)
    if dim_Hd != 1
        println("H_$d has dimension $dim_Hd, expected 1")
        return false
    end
    
    return true
end

"""
Compute the dimension of the i-th mod 2 homology group.
H_i = ker(∂_i) / im(∂_{i+1})
dim(H_i) = dim(ker(∂_i)) - dim(im(∂_{i+1}))
"""
function compute_mod2_homology_dim(complex, i::Int)
    # Get boundary matrices
    ∂_i = boundary_matrix_mod2(complex, i)
    ∂_ip1 = boundary_matrix_mod2(complex, i+1)
    
    # Compute dimensions
    if isempty(∂_i) || size(∂_i, 2) == 0
        dim_ker_i = 0
    else
        ker_basis = kernel_basis_mod2_sparse(∂_i)
        dim_ker_i = length(ker_basis)
    end
    
    if isempty(∂_ip1) || size(∂_ip1, 2) == 0
        dim_im_ip1 = 0
    else
        # dim(im) = rank of matrix mod 2
        dim_im_ip1 = rank_mod2(∂_ip1)
    end
    
    return dim_ker_i - dim_im_ip1
end

"""
Compute rank of a sparse matrix over GF(2).
"""
function rank_mod2(A)
    isempty(A) && return 0
    m, n = size(A)
    
    # Convert to BitMatrix
    R = falses(m, n)
    for j in 1:n
        for p in nzrange(A, j)
            i = rowvals(A)[p]
            if isodd(A.nzval[p])
                R[i, j] ⊻= true
            end
        end
    end
    
    # Row echelon form
    rank = 0
    row = 1
    for col in 1:n
        row > m && break
        
        # Find pivot
        piv = 0
        for r in row:m
            if R[r, col]
                piv = r
                break
            end
        end
        piv == 0 && continue
        
        # Swap
        if piv != row
            R[row, :], R[piv, :] = R[piv, :], R[row, :]
        end
        
        rank += 1
        
        # Eliminate
        for r in (row+1):m
            if R[r, col]
                R[r, :] .⊻= R[row, :]
            end
        end
        
        row += 1
    end
    
    return rank
end

"""
Construct the boundary matrix ∂_i: C_i → C_{i-1} over GF(2).
complex: Vector of facets (maximal simplices)
i: dimension

Returns a sparse matrix where:
- Rows are (i-1)-simplices
- Columns are i-simplices
- Entry [σ, τ] = 1 if σ is a face of τ (mod 2)
"""
function boundary_matrix_mod2(complex, i::Int)
    # Get all i-simplices and (i-1)-simplices
    simplices_i = get_simplices_of_dimension(complex, i)
    simplices_im1 = get_simplices_of_dimension(complex, i-1)
    
    n_im1 = length(simplices_im1)
    n_i = length(simplices_i)
    
    if n_i == 0 || n_im1 == 0
        return spzeros(Int, max(1, n_im1), max(1, n_i))
    end
    
    # Build index maps
    face_to_idx = Dict(sort(collect(σ)) => idx for (idx, σ) in enumerate(simplices_im1))
    
    rows = Int[]
    cols = Int[]
    
    for (j, τ) in enumerate(simplices_i)
        τ_sorted = sort(collect(τ))
        # Get all (i-1)-faces of τ
        for k in 1:length(τ_sorted)
            face = [τ_sorted[l] for l in 1:length(τ_sorted) if l != k]
            if haskey(face_to_idx, face)
                push!(rows, face_to_idx[face])
                push!(cols, j)
            end
        end
    end
    
    # Mod 2: just count appearances mod 2
    vals = ones(Int, length(rows))
    
    ∂ = sparse(rows, cols, vals, n_im1, n_i)
    
    # Make it mod 2
    for j in 1:size(∂, 2)
        for p in nzrange(∂, j)
            ∂.nzval[p] = ∂.nzval[p] % 2
        end
    end
    
    return ∂
end

"""
Get all i-dimensional simplices in the complex.
"""
function get_simplices_of_dimension(complex, i::Int)
    i < 0 && return Set{Vector{Int}}()
    
    simplices = Set{Vector{Int}}()
    
    for facet in complex
        facet_vec = sort(collect(facet))
        # Generate all i-subsets
        for subset in combinations(facet_vec, i+1)
            push!(simplices, subset)
        end
    end
    
    return collect(simplices)
end

# Helper: combinations
function combinations(arr, k)
    n = length(arr)
    k > n && return Vector{eltype(arr)}[]
    k == 0 && return [eltype(arr)[]]
    
    result = Vector{eltype(arr)}[]
    
    function generate(start, current)
        if length(current) == k
            push!(result, copy(current))
            return
        end
        
        for i in start:(n - k + length(current) + 1)
            push!(current, arr[i])
            generate(i + 1, current)
            pop!(current)
        end
    end
    
    generate(1, eltype(arr)[])
    return result
end

combinations (generic function with 1 method)

In [6]:
using BenchmarkTools


function kernel_basis_echelon_prioritize(B, S)
    """
    Convert kernel basis B to echelon form over GF(2), prioritizing columns in S.
    Returns (B_ech, pivots) where:
    - B_ech is the echelon basis
    - pivots[i] is the pivot position (leading 1) of B_ech[i]
    - Columns in S are processed first to maximize forced coefficients
    """
    isempty(B) && return (BitVector[], Int[])
    
    n = length(B[1])
    k = length(B)
    
    # Copy basis vectors
    B_ech = [copy(b) for b in B]
    pivots = Int[]
    
    # Build column order: S columns first, then others
    cols_in_S = [j for j in 1:n if S[j]]
    cols_not_in_S = [j for j in 1:n if !S[j]]
    col_order = vcat(cols_in_S, cols_not_in_S)
    
    current_row = 1
    for col in col_order
        current_row > k && break
        
        # Find a vector with 1 at position col (at or below current_row)
        piv = 0
        for r in current_row:k
            if B_ech[r][col]
                piv = r
                break
            end
        end
        
        piv == 0 && continue  # No pivot in this column
        
        # Swap to current position
        if piv != current_row
            B_ech[current_row], B_ech[piv] = B_ech[piv], B_ech[current_row]
        end
        
        push!(pivots, col)
        
        # Eliminate this column in all other rows
        for r in 1:k
            if r != current_row && B_ech[r][col]
                B_ech[r] .⊻= B_ech[current_row]
            end
        end
        
        current_row += 1
    end
    
    return (B_ech, pivots)
end


# function enumerate_kernel_with_constraints(A, B, S)
#     m, n = size(A)
#     results = BitVector[]
    
#     if isempty(B)
#         x = falses(n)
#         if all(.!S) && check_Ax_is_02(A, x)
#             push!(results, x)
#         end
#         return results
#     end
    
#     # Convert basis to echelon form, prioritizing S columns
#     B_ech, pivots = kernel_basis_echelon_prioritize(B, S)
#     k = length(B_ech)
    
#     # Determine forced and free coefficients
#     forced_lambda = falses(k)
#     free_indices = Int[]
    
#     for i in 1:k
#         piv = pivots[i]
#         if S[piv]
#             forced_lambda[i] = true
#         else
#             push!(free_indices, i)
#         end
#     end
    
#     # Compute base vector from forced coefficients
#     x_forced = falses(n)
#     for i in 1:k
#         if forced_lambda[i]
#             x_forced .⊻= B_ech[i]
#         end
#     end
    
#     # Early exit if x_forced doesn't satisfy S
#     for j in 1:n
#         if S[j] && !x_forced[j]
#             return results
#         end
#     end
    
#     num_free = length(free_indices)
    
#     # Early exit if no free variables
#     if num_free == 0
#         if check_Ax_is_02(A, x_forced)
#             push!(results, copy(x_forced))
#         end
#         return results
#     end
    
#     # Pre-allocate with size hint
#     sizehint!(results, min(2^num_free, 1000))
    
#     # Gray code enumeration
#     x = copy(x_forced)
    
#     # First iteration (all free vars = 0)
#     if check_Ax_is_02(A, x)
#         push!(results, copy(x))
#     end
    
#     # Enumerate using Gray code
#     for i in 1:(2^num_free - 1)
#         gray = i ⊻ (i >> 1)
#         gray_prev = (i - 1) ⊻ ((i - 1) >> 1)
#         changed_bit = trailing_zeros(gray ⊻ gray_prev) + 1
        
#         idx = free_indices[changed_bit]
#         x .⊻= B_ech[idx]
        
#         if check_Ax_is_02(A, x)
#             push!(results, copy(x))
#         end
#     end
    
#     return results
# end

# function check_Ax_is_02(A, x)
#     m, n = size(A)
#     result = zeros(Int, m)
    
#     for j in 1:n
#         if x[j]
#             for p in nzrange(A, j)
#                 i = rowvals(A)[p]
#                 result[i] += A.nzval[p]
                
#                 if result[i] > 2
#                     return false
#                 end
#             end
#         end
#     end
#     return true

# end

kernel_basis_echelon_prioritize (generic function with 1 method)

In [7]:
function enumerate_kernel_with_constraints(A, B, S)
    m, n = size(A)
    results = BitVector[]
    
    if isempty(B)
        x = falses(n)
        if all(.!S) && check_Ax_is_02(A, x)
            push!(results, x)
        end
        return results
    end
    
    # Convert basis to echelon form (using BitVectors)
    B_ech, pivots = kernel_basis_echelon_prioritize(B, S)
    k = length(B_ech)
    
    # Determine forced and free coefficients
    forced_lambda = falses(k)
    free_indices = Int[]
    
    for i in 1:k
        piv = pivots[i]
        if S[piv]
            forced_lambda[i] = true
        else
            push!(free_indices, i)
        end
    end
    
    # Compute base vector from forced coefficients (stays as BitVector)
    x_forced = falses(n)
    for i in 1:k
        if forced_lambda[i]
            x_forced .⊻= B_ech[i]
        end
    end
    
    # Early exit if x_forced doesn't satisfy S
    for j in 1:n
        if S[j] && !x_forced[j]
            return results
        end
    end
    
    num_free = length(free_indices)
    
    # Early exit if no free variables
    if num_free == 0
        if check_Ax_is_02(A, x_forced)
            push!(results, copy(x_forced))
        end
        return results
    end
    
    # Choose integer type based on num_free (not k or n!)
    if num_free <= 64
        return enumerate_with_int_encoding(A, B_ech, x_forced, free_indices, num_free, UInt64)
    elseif num_free <= 128
        return enumerate_with_int_encoding(A, B_ech, x_forced, free_indices, num_free, UInt128)
    else
        # Fall back to BitVector for large num_free
        return enumerate_with_bitvector(A, B_ech, x_forced, free_indices, num_free)
    end
end

function enumerate_with_int_encoding(A, B_ech, x_forced, free_indices, num_free, ::Type{IntType}) where IntType <: Unsigned
    n = length(x_forced)
    results = BitVector[]
    sizehint!(results, min(2^num_free, 1000))
    
    # Convert free basis vectors and x_forced to integers
    B_free_int = IntType[bitvector_to_uint(B_ech[idx], IntType) for idx in free_indices]
    x_forced_int = bitvector_to_uint(x_forced, IntType)
    
    x_int = x_forced_int
    
    # First iteration
    if check_Ax_is_02_int(A, x_int, n)
        push!(results, uint_to_bitvector(x_int, n))
    end
    
    # Gray code enumeration
    for i in 1:(2^num_free - 1)
        gray = i ⊻ (i >> 1)
        gray_prev = (i - 1) ⊻ ((i - 1) >> 1)
        changed_bit = trailing_zeros(gray ⊻ gray_prev) + 1
        
        # Fast integer XOR
        x_int ⊻= B_free_int[changed_bit]
        
        if check_Ax_is_02_int(A, x_int, n)
            push!(results, uint_to_bitvector(x_int, n))
        end
    end
    
    return results
end

function check_Ax_is_02_int(A, x_int::T, n::Int) where T <: Unsigned
    m = size(A, 1)
    result = zeros(Int, m)
    
    # Iterate over set bits
    x_temp = x_int
    bit_pos = 0
    
    while x_temp != 0
        tz = trailing_zeros(x_temp)
        bit_pos += tz
        j = bit_pos + 1
        
        if j <= n
            for p in nzrange(A, j)
                i = rowvals(A)[p]
                result[i] += A.nzval[p]
                
                if result[i] > 2
                    return false
                end
            end
        end
        
        x_temp >>= (tz + 1)
        bit_pos += 1
    end
    
    return true
end

function enumerate_with_bitvector(A, B_ech, x_forced, free_indices, num_free)
    results = BitVector[]
    sizehint!(results, min(2^num_free, 1000))
    
    x = copy(x_forced)
    
    if check_Ax_is_02(A, x)
        push!(results, copy(x))
    end
    
    for i in 1:(2^num_free - 1)
        gray = i ⊻ (i >> 1)
        gray_prev = (i - 1) ⊻ ((i - 1) >> 1)
        changed_bit = trailing_zeros(gray ⊻ gray_prev) + 1
        
        idx = free_indices[changed_bit]
        x .⊻= B_ech[idx]
        
        if check_Ax_is_02(A, x)
            push!(results, copy(x))
        end
    end
    
    return results
end

function check_Ax_is_02(A, x::BitVector)
    m, n = size(A)
    result = zeros(Int, m)
    
    for j in 1:n
        if x[j]
            for p in nzrange(A, j)
                i = rowvals(A)[p]
                result[i] += A.nzval[p]
                
                if result[i] > 2
                    return false
                end
            end
        end
    end
    return true
end

function bitvector_to_uint(bv::BitVector, ::Type{T}) where T <: Unsigned
    n = length(bv)
    result = T(0)
    for i in 1:min(n, 8 * sizeof(T))
        if bv[i]
            result |= T(1) << (i - 1)
        end
    end
    return result
end

function uint_to_bitvector(x::T, n::Int) where T <: Unsigned
    bv = falses(n)
    for i in 1:min(n, 8 * sizeof(T))
        if (x >> (i - 1)) & 1 == 1
            bv[i] = true
        end
    end
    return bv
end

uint_to_bitvector (generic function with 1 method)

In [8]:
using Profile
using BenchmarkTools
using ProgressMeter

global mat_DB_bin = open("rank_4_simple_bin_mat_DB_bin.jls", "r") do io
    deserialize(io)
end

global iso_DB = open("rank_4_iso_DB_m_7-15.jls", "r") do io
    deserialize(io)
end

function subset_bitvector(superset::Vector{UInt16}, subset::Vector{UInt16})
    n = length(superset)
    result = falses(n)
    
    j = 1  # index dans subset
    for i in 1:n
        if j <= length(subset) && superset[i] == subset[j]
            result[i] = true
            j += 1
        end
    end
    
    return result
end




function build_finalDB_single_v!(pseudo_manifolds_DB::Dict{Int,Vector{Set{BitVector}}},mat_DB::Dict{Int,Vector{Vector{UInt16}}},iso_DB::Dict{Int,Dict{Int,Tuple{Int,Int,Any}}},mmax)
    mmin = minimum(collect(keys(mat_DB)))
    for m=mmin:mmax
        println(m)
        pseudo_manifolds_DB[m] = Vector{Set{BitVector}}()
        for (l,bases) in enumerate(mat_DB[m])
            # display(bases)
            V = reduce(|,bases)
            compl_bases = [base⊻V for base in bases] # we need to complement to get the correct boundary matrix
            ridges, A = boundary_incidence_facets_to_ridges(compl_bases)  
            basis = kernel_basis_mod2_sparse(A)
            push!(pseudo_manifolds_DB[m], Set{BitVector}())
            if m==mmin
                mandatory_facets_bit=falses(length(bases))
                all_solutions_bit = kernel_elements_with_Ax_in_0_or_2(A,basis)
                for K_bit in all_solutions_bit
                    K = simplicial_complex(collect([i for i=1:16 if facet>>(i-1)&1==1] for facet in compl_bases[findall(K_bit)]))
                    if is_sphere(K)
                        push!(pseudo_manifolds_DB[m][l],K_bit)
                    end
                end
            else
                index_contraction, v_contract, perm = iso_DB[m][l]
                @showprogress desc="Number of links $(length(pseudo_manifolds_DB[m-1][index_contraction]))" for L in pseudo_manifolds_DB[m-1][index_contraction]
                    mandatory_facets = [reduce(|,[UInt16(1)<<(perm[i]-1) for i=1:(sizeof(facet_L)) if (facet_L>>(i-1)&1)==1],init=UInt16(0))⊻(UInt16(1)<<(v_contract-1)) for facet_L in mat_DB[m-1][index_contraction][findall(L)]]
                    # print(mandatory_facets)
                    mandatory_facets_bit = subset_bitvector(bases, mandatory_facets)
                    t1 = time()
                    all_solutions_bit = enumerate_kernel_with_constraints(A,basis,mandatory_facets_bit)
                    if (time() - t1)> 1
                        println("Enum: ", time() - t1, " seconds")
                    end
                    if m<9
                        for K_bit in all_solutions_bit
                            K = simplicial_complex(collect([i for i=1:16 if facet>>(i-1)&1==1] for facet in compl_bases[findall(K_bit)]))
                            if is_homology_sphere(K)
                                push!(pseudo_manifolds_DB[m][l],K_bit)
                            end
                        end
                    else
                        union!(pseudo_manifolds_DB[m][l],all_solutions_bit)
                    end
                end
            end
        end
    end
end


global pseudo_manifolds_DB = Dict{Int,Vector{Set{BitVector}}}()

build_finalDB_single_v!(pseudo_manifolds_DB,mat_DB_bin,iso_DB,15)

# using ProfileView
# ProfileView.view()




6
7


Number of links 36 100%|█████████████████████████████████| Time: 0:00:29
Number of links 15 100%|█████████████████████████████████| Time: 0:00:00
Number of links 15 100%|█████████████████████████████████| Time: 0:00:02
Number of links 36 100%|█████████████████████████████████| Time: 0:00:02
Number of links 15 100%|█████████████████████████████████| Time: 0:00:01


8


Number of links 5 100%|██████████████████████████████████| Time: 0:00:06
Number of links 11 100%|█████████████████████████████████| Time: 0:06:41


9


...

Enum: 65.40093398094177 seconds


Number of links 7130   1%|▍                              |  ETA: 5 days, 10:36:44

Enum: 64.7468090057373 seconds


Number of links 7130   1%|▍                              |  ETA: 5 days, 10:35:09

LoadError: InterruptException: